<a href="https://colab.research.google.com/github/ernestoaguaysol-unpaz/sistemas-inteligentes-2026/blob/main/03_redes_convolucionales/001_pizza_carne_red_prealimentada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Dataset https://drive.google.com/file/d/1O5jW6N30YCUZEAXtWupgOwQmxBHIgWyh/view
# Crear directorio data y guardarlo ahí.

In [ ]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

import tensorflow as tf

from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPool2D, Flatten, Rescaling

In [ ]:
# Constantes

BATCH_SIZE = 32
IMG_HEIGHT = 256
IMG_WIDTH = 256

DIR_TRAIN = "data/pizza_steak/train/"
DIR_TEST = "data/pizza_steak/test/"

In [ ]:
# Generar un Dataset con el directorio de archivos de train
# Función image_dataset_from_directory
# Lee datos del disco en batches, evitando colapsar la RAM si está todo en memoria.
dataset_train = tf.keras.preprocessing.image_dataset_from_directory(
    # Directorio
    DIR_TRAIN,

    # Etiquetas inferidas desde los nombres de directorios
    labels='inferred',

    # Clasificación binaria
    label_mode='binary',

    batch_size=BATCH_SIZE,
    image_size=(IMG_WIDTH, IMG_HEIGHT),

    # Por defecto mezcla cada vez que se toman al azar BATCH_SIZE elementos del directorio.
    # Para obtener métricas es mejor no mezclar.
    shuffle=False
)

print("Clases dataset entrenamiento: ", dataset_train.class_names)

In [ ]:
# Veamos algunas imágenes de entrenamiento
# Como shuffle=False se leerá progresivamente las pizzas
# Para obtener ejemplos de distintas clases se puede poner temporalmente shuffle=True
plt.figure(figsize=(10, 10))
for images, labels in dataset_train.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        img_indice = random.randint(0, len(images)-1)
        plt.imshow(images[img_indice].numpy().astype("uint8"))
        plt.title(dataset_train.class_names[int(labels[img_indice])])
        plt.axis("off")

In [ ]:
# Hacemos lo mismo para generar el dataset de test
dataset_test = tf.keras.preprocessing.image_dataset_from_directory(
    DIR_TEST,
    labels='inferred',
    label_mode='binary',
    batch_size=BATCH_SIZE,
    image_size=(IMG_WIDTH, IMG_HEIGHT),
    shuffle=False
)

print("Clases dataset prueba: ", dataset_test.class_names)

In [ ]:
# Veamos algunas imágenes de test
plt.figure(figsize=(10, 10))
for images, labels in dataset_test.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        img_indice = random.randint(0, len(images)-1)
        plt.imshow(images[img_indice].numpy().astype("uint8"))
        plt.title(dataset_test.class_names[int(labels[img_indice])])
        plt.axis("off")

In [ ]:
# Creando el modelo

modelo = tf.keras.Sequential([
    Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3)),
    Rescaling(1./255),
    Flatten(),
    Dense(10, activation="relu"),
    Dense(20, activation="relu"),
    Dense(10, activation="relu"),
    Dense(units=1, activation="sigmoid")
])

In [ ]:
modelo.summary()

In [ ]:
modelo.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)

In [ ]:
# La función image_dataset_from_directory genera un objeto tf.data.Dataset
# A .fit se le puede pasar el tf.data.Dataset (sin explicitar X e y)
entrenamiento_info = modelo.fit(
    dataset_train,
    epochs=10,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
# Ploteos del entrenamiento del modelo
fig, axs = plt.subplot_mosaic([
    ['perdida', 'exactitud']
],
    layout='constrained',
    figsize=(8, 3)
)

axs['perdida'].set_title('Curva de entrenamiento - Pérdida')
axs['perdida'].plot(entrenamiento_info.history['loss'])
axs['perdida'].grid()
axs['perdida'].set_ylim(-0.1, 5)

axs['exactitud'].set_title('Curva de entrenamiento - Exactitud')
axs['exactitud'].plot(entrenamiento_info.history['accuracy'])
axs['exactitud'].grid()
axs['exactitud'].set_ylim(-0.1, 1.1)
axs['exactitud'].set_yticks(np.arange(0, 1.05, 0.10))

plt.suptitle("Entrenamiento del modelo")
plt.show()

In [ ]:
def plot_matriz_confusion(y_real,
                          y_pred,
                          labels,
                          figsize=(4, 4),
                          nombre=''):
    np.set_printoptions(suppress=True)
    sns.set_theme(font_scale=1.5)
    matriz_confusion = confusion_matrix(y_real, y_pred)

    fig, ax = plt.subplots(figsize=figsize)

    ax = sns.heatmap(
        matriz_confusion,
        cbar=False,
        annot=True,
        fmt='g'
    )

    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticklabels(labels, rotation=0)

    plt.suptitle(f"Matriz de confusión - {nombre}")
    plt.xlabel("Predicción")
    plt.ylabel("Etiqueta real")

    plt.show()
    sns.set_theme(font_scale=1)

In [ ]:
np.set_printoptions(suppress=True)
y_train = tf.concat([y for x, y in dataset_train], axis=0)
y_train_pred_proba = modelo.predict(dataset_train)
y_train_pred = tf.round(y_train_pred_proba)
plot_matriz_confusion(y_train, y_train_pred, labels=dataset_train.class_names, nombre='Conj. de entrenamiento')

In [ ]:
np.set_printoptions(suppress=True)
y_test = tf.concat([y for x, y in dataset_test], axis=0)
y_test_pred_proba = modelo.predict(dataset_test)
y_test_pred = tf.round(y_test_pred_proba)
plot_matriz_confusion(y_test, y_test_pred, labels=dataset_test.class_names, nombre='Conj. de pruebas')

In [ ]:
# Función para preprocesar cualquier imagen para el modelo
def preprocesar_imagen(nombre_archivo, img_shape=256):
    """
    Lee la imagen con el nombre de archivo,
    y arma un tensor con la forma (img_shape, img_shape, color_channels)
    """

    # Leer el archivo
    img = tf.io.read_file(nombre_archivo)

    # Convertir la imagen a tensor
    img = tf.image.decode_image(img)

    # Redimensionar la imagen a la forma
    img = tf.image.resize(img, size=[img_shape, img_shape])

    # Reescalar la imagen para tener los valores entre 0 y 1
    img = img/255.

    plt.grid(False)
    plt.imshow(img)
    plt.show()

    return img

In [ ]:
imagen = random.sample(os.listdir(DIR_TRAIN+"pizza"), 1)[0]
imagen = preprocesar_imagen(DIR_TRAIN+"pizza/" + imagen)
batch_imagenes = tf.expand_dims(imagen, axis=0)
ejemploFotoPrediccion = modelo.predict(batch_imagenes)

print("Predicción: ", dataset_train.class_names[int(tf.round(ejemploFotoPrediccion))])
print("Predicción prob. ", ejemploFotoPrediccion)